# nuScenes Dino: notebook quickstart

This notebook demonstrates how to wire nuScenes RGB frames into the DINOv3 encoder. It includes a small dry-run that works without the dataset plus a ready-to-run section you can execute once your data and weights are in place.

## Setup

* Activate your Conda env and install the project in editable mode (`pip install -e .`).
* Download and extract the nuScenes dataset, then set `dataroot` to that path.
* Log into Hugging Face if the DINOv3 checkpoint you use is gated.

In [7]:
from pathlib import Path

dataroot = Path('../../../data/sets/nuscenes')
print(f'Using dataset at: {dataroot}')

Using dataset at: ../../../data/sets/nuscenes


## Dry-run without nuScenes

Create a synthetic `SensorBatch` so you can validate shapes and basic tensor handling even if you do not have the dataset downloaded yet.

In [ ]:
from PIL import Image
import numpy as np

palette = {
    'CAM_FRONT': (255, 48, 48),
    'CAM_FRONT_RIGHT': (64, 160, 255),
    'CAM_FRONT_LEFT': (80, 200, 120),
    'CAM_BACK': (200, 120, 255),
    'CAM_BACK_LEFT': (255, 196, 92),
    'CAM_BACK_RIGHT': (120, 120, 120),
}

camera_images = {name: Image.new('RGB', (64, 64), color) for name, color in palette.items()}
print('Token: demo-sample')
print('Camera channels:')
for name, image in camera_images.items():
    print(f'  {name}: size={image.size}, mode={image.mode}')

Token: demo-sample
Camera channels:
  CAM_FRONT: size=(64, 64), mode=RGB
  CAM_FRONT_RIGHT: size=(64, 64), mode=RGB
  CAM_FRONT_LEFT: size=(64, 64), mode=RGB
  CAM_BACK: size=(64, 64), mode=RGB
  CAM_BACK_LEFT: size=(64, 64), mode=RGB
  CAM_BACK_RIGHT: size=(64, 64), mode=RGB


In [3]:
stack = np.stack([np.asarray(img).transpose(2, 0, 1) for img in camera_images.values()]) / 255.0
print('Tensor layout:', stack.shape)
print('Per-channel means per camera:')
print(stack.mean(axis=(2, 3)))

Tensor layout: (6, 3, 64, 64)
Per-channel means per camera:
[[1.         0.18823529 0.18823529]
 [0.25098039 0.62745098 1.        ]
 [0.31372549 0.78431373 0.47058824]
 [0.78431373 0.47058824 1.        ]
 [1.         0.76862745 0.36078431]
 [0.47058824 0.47058824 0.47058824]]


## Run with nuScenes + DINOv3

Uncomment and execute the following once your dataset and weights are available.
It streams camera frames, feeds them to the DINOv3 encoder, and returns pooled features
ready for a language model.

In [8]:
from nuscenes_dino import DinoImageEncoder, NuScenesPipeline

pipeline = NuScenesPipeline(
    dataroot=str(dataroot),
    version='v1.0-mini',
    include_radar=False,
    include_lidar=False,
)
encoder = DinoImageEncoder(device_map='auto')

first_batch = next(pipeline.stream())
features = encoder.encode(first_batch.camera_images.values())
print('Pixel tensor shape:', features['pixel_values'].shape)
print('Last hidden state:', tuple(features['last_hidden_state'].shape))
print('Pooled output:', tuple(features['pooled_output'].shape))


Pixel tensor shape: torch.Size([6, 3, 224, 224])
Last hidden state: (6, 50, 768)
Pooled output: (6, 768)
